In [43]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/NodeA7/FlyRank_AI"
REPO_DIR = "FlyRank_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Starter CSV not found."
print("Starter data ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/FlyRank_AI/FlyRank_AI/FlyRank_AI
Starter data ready.


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodeA7/FlyRank_AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I’ll review a page when it has been unchanged for at least 90 days and had at least 300 impressions in the last 90 days. Among those pages, more impressions means a higher score.

The two checks below are descriptive. Staleness is tied to FlyRank refresh flags; volume is tied to the quick-win idea. Verdicts describe these data, not cause-and-effect.

In [44]:
from pathlib import Path
import pandas as pd

# Load the CSV file
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / "data/raw/content_refresh_anonymized.csv" for root in roots]
candidates += [Path("/content/FlyRank_AI/data/raw/content_refresh_anonymized.csv"),
               Path("/content/flyrank-ai/data/raw/content_refresh_anonymized.csv")]
csv_path = next((p for p in candidates if p.is_file()), None)


usecols = ["content_id", "content_type", "impressions_90d", "clicks_90d",
           "days_since_last_update", "avg_position"]

# Read dataset and clean up missing/null values
df = pd.read_csv(csv_path, usecols=usecols)

# Handle missing values: fill numeric traffic with 0, average position with 0 (meaning unranked), and staleness with a safe default
df["impressions_90d"] = df["impressions_90d"].fillna(0).astype(int)
df["clicks_90d"] = df["clicks_90d"].fillna(0).astype(int)
df["days_since_last_update"] = df["days_since_last_update"].fillna(9999).astype(int)
df["avg_position"] = df["avg_position"].fillna(0.0)
df["content_type"] = df["content_type"].fillna("unknown")

df["stale_bucket"] = pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, float("inf")],
                            labels=["0-30", "31-90", "91-180", "181+"])
df["volume_bucket"] = pd.cut(df["impressions_90d"], [-1, 99, 499, 2999, float("inf")],
                             labels=["<100", "100-499", "500-2,999", "3,000+"])

stale_table = df.groupby("stale_bucket", observed=False).agg(
    n=("content_id", "size"), median_impressions=("impressions_90d", "median"),
    median_clicks=("clicks_90d", "median")).reset_index()
volume_table = df.groupby("volume_bucket", observed=False).agg(
    n=("content_id", "size"), median_impressions=("impressions_90d", "median"),
    median_clicks=("clicks_90d", "median")).reset_index()
print("Staleness (days since update) — bucket counts and observed traffic:")
print(stale_table.to_string(index=False))
print("Volume (trailing 90-day impressions) — bucket counts and observed traffic:")
print(volume_table.to_string(index=False))

Staleness (days since update) — bucket counts and observed traffic:
stale_bucket     n  median_impressions  median_clicks
        0-30 20480               470.0            1.0
       31-90   175               510.0            0.0
      91-180  9171              1692.0            2.0
        181+   174                15.5            0.0
Volume (trailing 90-day impressions) — bucket counts and observed traffic:
volume_bucket    n  median_impressions  median_clicks
         <100 7994                12.0            0.0
      100-499 5280               251.5            0.0
    500-2,999 8443              1230.0            2.0
       3,000+ 8283              8426.0           20.0


**Signal verdicts**

- **Staleness — MIXED.** The 91–180 day bucket has a higher median impression count than the 0–30 day bucket, but the 181+ bucket is small and has very low median impressions. Old does not always mean worth a refresh. I use 90 days as a review cutoff, not a claim that staleness causes decline.
- **Volume — CONFIRMED.** Higher impression buckets have more clicks in this snapshot. That makes impressions a useful way to rank visible pages. It does not show that an edit will help.

## 2. Build the ranked queue

The CSV includes pseudonymous content IDs so a reviewer can match a row inside the project. Do not publish the queue. The notebook recreates it at `work/outputs/baseline_action_score.csv` each time it runs.

In [45]:
# One rule: pages at least 90 days since update AND at least 300 impressions in 90 days.
# Score is current impressions for qualifying pages; otherwise zero. No fitted weights.
df["score"] = (df["days_since_last_update"].ge(90) & df["impressions_90d"].ge(300)).astype(int) * df["impressions_90d"]
df["reason_code"] = "not_qualified"
df.loc[df["score"].gt(0), "reason_code"] = "stale_and_visible"
df["action"] = "monitor"
df.loc[df["score"].gt(0), "action"] = "review_refresh"
queue = df[["content_id", "score", "reason_code", "action", "days_since_last_update", "impressions_90d", "clicks_90d", "avg_position"]].copy()
queue = queue.sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
queue.insert(0, "rank", range(1, len(queue) + 1))

repo_root = Path("/content/FlyRank_AI")
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Queue rows: {len(queue):,} | review_refresh: {(queue['action'] == 'review_refresh').sum():,}")
print(f"Wrote: {output_path}")
print("Score check: top score =", int(queue.loc[0, "score"]))

Queue rows: 30,000 | review_refresh: 7,234
Wrote: /content/FlyRank_AI/work/outputs/baseline_action_score.csv
Score check: top score = 517715


In [46]:
!git config user.name "Node_A7"
!git config user.email "zaksababras@gmail.com"

## 3. Top-20 review

Each line says the action, why the page is in the queue, and what could make this pick wrong. These are review prompts, not claims that a refresh will work.

In [47]:
# Show 20 rows with no IDs, then write one short skeptic's note for each.
top20 = queue.head(20).copy()
for row in top20.itertuples(index=False):
    rank = row.rank
    if row.avg_position > 0 and row.avg_position <= 20:
      wrong = "It may already rank well."
    elif row.clicks_90d == 0:
      wrong = "The issue may not be the content."
    else:
      wrong = "An update may not improve performance."

    print(f"{rank}. Action: {row.action}; why: {row.reason_code}, \n{int(row.days_since_last_update)} days since update and {int(row.impressions_90d):,} impressions; \nwhat would make it wrong: {wrong}\n")

1. Action: review_refresh; why: stale_and_visible, 
104 days since update and 517,715 impressions; 
what would make it wrong: It may already rank well.

2. Action: review_refresh; why: stale_and_visible, 
104 days since update and 443,434 impressions; 
what would make it wrong: An update may not improve performance.

3. Action: review_refresh; why: stale_and_visible, 
104 days since update and 347,399 impressions; 
what would make it wrong: It may already rank well.

4. Action: review_refresh; why: stale_and_visible, 
104 days since update and 309,910 impressions; 
what would make it wrong: It may already rank well.

5. Action: review_refresh; why: stale_and_visible, 
104 days since update and 309,192 impressions; 
what would make it wrong: It may already rank well.

6. Action: review_refresh; why: stale_and_visible, 
104 days since update and 295,097 impressions; 
what would make it wrong: It may already rank well.

7. Action: review_refresh; why: stale_and_visible, 
104 days since up

## 4. Weak picks + leakage check

A high impression count can still be a weak reason to refresh. The checks below surface examples that already rank near the top or have no clicks. The rule uses only current staleness and current impressions; it does not use trend labels or future outcomes.

In [48]:
# Show weak-pick examples from the top 20 and confirm the rule has no forbidden inputs.
weak = top20[(top20["avg_position"].between(1, 20)) | (top20["clicks_90d"].eq(0))]
print("Weak-pick examples in top 20:", len(weak))
for row in weak.head(5).itertuples(index=False):
    print(f"Rank {row.rank}: {int(row.impressions_90d):,} impressions, {int(row.clicks_90d):,} \
            clicks, position {row.avg_position:g}; check whether visibility is valuable and measurement is sound.")
used_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label", "future_impressions", "future_clicks"}
assert used_inputs.isdisjoint(forbidden)
print("Leakage check: score uses only current staleness and current impressions; no trend label or future-window field is used.")

Weak-pick examples in top 20: 14
Rank 1: 517,715 impressions, 741             clicks, position 4.2; check whether visibility is valuable and measurement is sound.
Rank 3: 347,399 impressions, 1,854             clicks, position 4.2; check whether visibility is valuable and measurement is sound.
Rank 4: 309,910 impressions, 492             clicks, position 5.6; check whether visibility is valuable and measurement is sound.
Rank 5: 309,192 impressions, 2,689             clicks, position 2; check whether visibility is valuable and measurement is sound.
Rank 6: 295,097 impressions, 154             clicks, position 7.3; check whether visibility is valuable and measurement is sound.
Leakage check: score uses only current staleness and current impressions; no trend label or future-window field is used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.